# Safe Human-in-the-Loop RL on Google Colab

This notebook will guide you through setting up and running the Safe Human-in-the-Loop RL project on Google Colab with GPU support.

**Important:** Make sure to enable GPU runtime:
- Go to `Runtime` → `Change runtime type` → Select `GPU` (T4, V100, or A100)

---

## Step 1: Check GPU Availability

In [ ]:
import torch
import subprocess

# Check if GPU is available
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
else:
    print("⚠️ WARNING: GPU not available. Please enable GPU in Runtime settings.")

# Check nvidia-smi
!nvidia-smi

## Step 2: Install System Dependencies

Install required system packages for SMARTS simulator.

In [ ]:
%%bash
# Update package list
apt-get update -qq

# Install system dependencies for SMARTS
apt-get install -y -qq \
    libspatialindex-dev \
    xorg \
    libx11-dev \
    libglu1-mesa-dev \
    libgl1-mesa-dev \
    libxrandr-dev \
    libxxf86vm-dev \
    libxcursor-dev \
    libxi-dev \
    libxinerama-dev \
    libxrender-dev \
    mesa-utils \
    xvfb \
    ffmpeg

echo "✓ System dependencies installed successfully!"

## Step 3: Clone the Repository

In [ ]:
import os

# Set working directory
WORKSPACE = "/content"
os.chdir(WORKSPACE)

# Clone the Safe-HIL-RL repository
if not os.path.exists("Safe-Human-in-the-Loop-RL"):
    !git clone https://github.com/OscarHuangWind/Safe-Human-in-the-Loop-RL.git
    print("✓ Repository cloned successfully!")
else:
    print("✓ Repository already exists!")

os.chdir("Safe-Human-in-the-Loop-RL")
print(f"Current directory: {os.getcwd()}")

## Step 4: Install PyTorch with CUDA Support

Install PyTorch compatible with Colab's CUDA version.

In [ ]:
# Check CUDA version and install appropriate PyTorch
import torch

cuda_version = torch.version.cuda
print(f"Current CUDA version: {cuda_version}")

# Install PyTorch (Colab usually has CUDA 11.8 or 12.x)
# We'll install a compatible version
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Verify installation
import torch
print(f"\n✓ PyTorch {torch.__version__} installed")
print(f"✓ CUDA available: {torch.cuda.is_available()}")

## Step 5: Install Python Dependencies from environment.yml

In [ ]:
# Install conda (miniconda) if not available
import sys
import os

# Since Colab doesn't use conda by default, we'll extract dependencies from environment.yml
# and install them with pip

if os.path.exists("environment.yml"):
    print("Found environment.yml, extracting dependencies...")
    !cat environment.yml
else:
    print("⚠️ environment.yml not found. Installing common dependencies...")

# Install common dependencies for RL projects
!pip install -q numpy pandas matplotlib seaborn gym gymnasium pyyaml tensorboard
!pip install -q opencv-python pillow scipy scikit-learn

print("\n✓ Python dependencies installed!")

## Step 6: Clone and Install SMARTS Simulator

In [ ]:
os.chdir(WORKSPACE)

# Clone SMARTS repository
if not os.path.exists("SMARTS"):
    print("Cloning SMARTS repository...")
    !git clone https://github.com/huawei-noah/SMARTS.git
    print("✓ SMARTS cloned successfully!")
else:
    print("✓ SMARTS already exists!")

os.chdir("SMARTS")

# Checkout to comp-1 branch (as specified in the guide)
print("\nChecking out comp-1 branch...")
!git checkout comp-1

print(f"\nCurrent directory: {os.getcwd()}")
print(f"Current branch: ", end="")
!git branch --show-current

## Step 7: Install SMARTS Dependencies

In [ ]:
# Install SMARTS with required extras
print("Installing SMARTS (this may take a few minutes)...")

# Install base SMARTS with camera_obs, test, and train extras
!pip install -q -e '.[camera_obs,test,train]'

# Install extra dependencies
!pip install -q -e '.[extras]'

print("\n✓ SMARTS installed successfully!")

# Verify SMARTS installation
try:
    import smarts
    print(f"✓ SMARTS version: {smarts.__version__}")
except ImportError as e:
    print(f"⚠️ Warning: Could not import SMARTS: {e}")

## Step 8: Build the Scenario

In [ ]:
# Go back to the Safe-HIL-RL directory
os.chdir(f"{WORKSPACE}/Safe-Human-in-the-Loop-RL")

print("Building scenario...")
print(f"Current directory: {os.getcwd()}")

# Check if scenario directory exists
if os.path.exists("scenario/straight/"):
    !scl scenario build --clean scenario/straight/
    print("\n✓ Scenario built successfully!")
else:
    print("⚠️ Warning: scenario/straight/ directory not found")
    print("Available directories:")
    !ls -la

## Step 9: Setup Virtual Display (for Visualization)

Since Colab doesn't have a display, we'll set up a virtual display using Xvfb.

In [ ]:
# Install pyvirtualdisplay for headless rendering
!pip install -q pyvirtualdisplay

from pyvirtualdisplay import Display

# Start virtual display
display = Display(visible=0, size=(1400, 900))
display.start()

print("✓ Virtual display started!")
print(f"Display: {display}")

## Step 10: Configure the Project

Check and modify configuration files if needed.

In [ ]:
# Check if config.yaml exists
if os.path.exists("config.yaml"):
    print("config.yaml found:")
    !cat config.yaml
else:
    print("⚠️ config.yaml not found")

# Check if main.py exists
if os.path.exists("main.py"):
    print("\n✓ main.py found")
    print("\nFirst 50 lines of main.py:")
    !head -50 main.py
else:
    print("⚠️ main.py not found")

# List all Python files
print("\n\nAvailable Python files:")
!find . -name "*.py" -type f | head -20

## Step 11: Fix System Paths in main.py

Update the sys.path in main.py to work with Colab environment.

In [ ]:
import sys

# Add SMARTS to Python path
smarts_path = f"{WORKSPACE}/SMARTS"
if smarts_path not in sys.path:
    sys.path.insert(0, smarts_path)
    print(f"✓ Added {smarts_path} to sys.path")

# Add Safe-HIL-RL to Python path
hil_path = f"{WORKSPACE}/Safe-Human-in-the-Loop-RL"
if hil_path not in sys.path:
    sys.path.insert(0, hil_path)
    print(f"✓ Added {hil_path} to sys.path")

print("\nCurrent sys.path:")
for p in sys.path[:5]:
    print(f"  - {p}")

## Step 12: Training Mode

Run the training script. Note: Human guidance features (keyboard/G29) won't work in Colab, but you can train the base models.

In [ ]:
# Make sure we're in the right directory
os.chdir(f"{WORKSPACE}/Safe-Human-in-the-Loop-RL")

# Run training
print("Starting training...")
print("Note: This will run in headless mode. Human guidance features are disabled in Colab.\n")

# Run main.py
!python main.py

## Step 13: Evaluation Mode

To run evaluation, first modify config.yaml to set mode to 'evaluation'.

In [ ]:
import yaml

# Load config.yaml
if os.path.exists("config.yaml"):
    with open("config.yaml", 'r') as f:
        config = yaml.safe_load(f)
    
    # Change mode to evaluation
    if config and 'mode' in config:
        config['mode'] = 'evaluation'
        
        # Save modified config
        with open("config.yaml", 'w') as f:
            yaml.dump(config, f)
        
        print("✓ Config updated to evaluation mode")
        print("\nUpdated config.yaml:")
        !cat config.yaml
    else:
        print("⚠️ Could not find 'mode' in config.yaml")
else:
    print("⚠️ config.yaml not found")

In [ ]:
# Run evaluation
print("Starting evaluation...\n")
!python main.py

## Step 14: Visualization and Results

View training logs, tensorboard, and saved models.

In [ ]:
# Check for output directories
print("Checking for results...\n")

# Common output directories
output_dirs = ['logs', 'results', 'models', 'checkpoints', 'runs', 'outputs']

for dir_name in output_dirs:
    if os.path.exists(dir_name):
        print(f"✓ Found {dir_name}/:")
        !ls -lh {dir_name}
        print()

# List all directories
print("\nAll directories in current path:")
!ls -la

In [ ]:
# Load TensorBoard if logs exist
%load_ext tensorboard

# Try to find tensorboard logs
log_dirs = ['logs', 'runs', 'tensorboard']
log_dir = None

for d in log_dirs:
    if os.path.exists(d):
        log_dir = d
        break

if log_dir:
    print(f"Starting TensorBoard with log directory: {log_dir}")
    %tensorboard --logdir {log_dir}
else:
    print("⚠️ No tensorboard logs found. Train the model first.")

## Step 15: Download Results to Local Machine

In [ ]:
from google.colab import files
import shutil

# Create a zip file of results
print("Creating archive of results...")

# Zip the results
if os.path.exists("logs") or os.path.exists("models") or os.path.exists("results"):
    !zip -r results.zip logs/ models/ results/ checkpoints/ 2>/dev/null || true
    
    if os.path.exists("results.zip"):
        print("\n✓ Results archived successfully!")
        print("Downloading results.zip...")
        files.download('results.zip')
    else:
        print("⚠️ No results to archive")
else:
    print("⚠️ No results directories found")

## Important Notes for Colab Usage

### Limitations:
1. **Human Guidance**: Keyboard control and G29 steering wheel won't work in Colab (no interactive input)
2. **Visualization**: Envision visualization (http://localhost:8081) won't be accessible directly
3. **Session Timeout**: Colab sessions timeout after inactivity. Save checkpoints frequently!
4. **GPU Time Limits**: Free Colab has GPU time limits. Consider Colab Pro for longer training.

### Workarounds:
1. **Training**: You can train base models (without human guidance) successfully
2. **Evaluation**: Load pre-trained models and evaluate them
3. **Results**: Download trained models and logs to continue locally
4. **Visualization**: Use TensorBoard for training metrics

### Saving Your Work:
```python
# Mount Google Drive to save results persistently
from google.colab import drive
drive.mount('/content/drive')

# Copy results to Drive
!cp -r logs/ /content/drive/MyDrive/SafeHIL_Results/
!cp -r models/ /content/drive/MyDrive/SafeHIL_Results/
```

### Reconnecting After Disconnect:
If your connection breaks, simply re-run the notebook from Step 1. Your files will be lost unless saved to Google Drive.

---

## Troubleshooting

**Issue**: CUDA out of memory
- **Solution**: Reduce batch size in config.yaml

**Issue**: SMARTS installation fails
- **Solution**: Check system dependencies in Step 2

**Issue**: Module not found errors
- **Solution**: Re-run Step 11 to fix Python paths

**Issue**: Scenario build fails
- **Solution**: Check if scenario/straight/ directory exists

---

## Next Steps

1. ✅ Train base models in Colab
2. ✅ Download trained models
3. ✅ Continue with human guidance on local machine with GPU
4. ✅ Use Colab for hyperparameter tuning and experiments
